In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/accidents_clean - accidents_clean.csv.csv')

print(df.shape)
print(df.columns.tolist())

(214044, 52)
['ID', 'Source', 'Severity', 'Start_Time', 'End_Time', 'Start_Lat', 'Start_Lng', 'Distance(mi)', 'Description', 'Street', 'City', 'County', 'State', 'Zipcode', 'Timezone', 'Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'Railway', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Sunrise_Sunset', 'High_Sev', 'Region', 'Is_Highway', 'Traffic_Control', 'State_Grp', 'Year', 'Month', 'Hour', 'DayOfWeek', 'Is_Weekend', 'Time_of_Day', 'Is_Rush_Hour', 'Precip_Missing', 'Is_Night', 'Wind_Direction_Clean', 'Weather_Group', 'Bad_Weather', 'Thin_Cell']


In [ ]:
#OVERALL STATISTICS

#count how many accidents belong to each severity level
print(df['Severity'].value_counts().sort_index())

#calculate the percentage of accidents belonging to each severity level
#convert counts into proportions into percentages
print(
    df['Severity'].value_counts(
        normalize=True
    ).sort_index() * 100
)

#count how many accidents have High_Sev = 0 or 1
#0 = Severity 1 or 2 (lower traffic-impact)
#1 = Severity 3 or 4 (higher traffic-impact)
print(df['High_Sev'].value_counts())

#count total number of accident records belonging to each region
print(df['Region'].value_counts())

Severity
1      2203
2    138966
3     71945
4       930
Name: count, dtype: int64
Severity
1     1.029228
2    64.924034
3    33.612248
4     0.434490
Name: proportion, dtype: float64
High_Sev
0    141169
1     72875
Name: count, dtype: int64
Region
South        104326
West          54628
Midwest       28738
Northeast     26352
Name: count, dtype: int64


In [ ]:
#WEATHER AND ROAD FACTOR PERCENTAGES

#define resuable function that analyze categorical factor
def analyze_factor(df, factor):
  return (
      df.groupby(factor)['High_Sev'].agg( #group accidents according to selected factor, for each group analyze High_Sev column
          Accident_Count='count', #count number of accident records in each group
          High_Severity_Count='sum', #add together all High_Sev values, sum gives us number of high-severity
          High_Severity_Rate='mean' #calculate average, average represents proportion of high-severity accidents within that group
      )
      .assign( #convert high-severity proportion into percentage
          High_Severity_Rate=lambda x:
          x['High_Severity_Rate'] * 100
      ).round(2)
  )

In [ ]:
#analyze weather conditions across entire dataset
print(analyze_factor(df, 'Weather_Group'))

               Accident_Count  High_Severity_Count  High_Severity_Rate
Weather_Group                                                         
Clear/Fair              87259                28005               32.09
Cloudy                  96620                33331               34.50
Fog/Low vis              5014                 1521               30.34
Rain                    14942                 5761               38.56
Snow/Ice                 3364                 1476               43.88
Thunderstorm             2217                  950               42.85
Unknown                  4628                 1831               39.56


In [ ]:
#compare highway & non-highway accidents
print(analyze_factor(df, 'Is_Highway'))

            Accident_Count  High_Severity_Count  High_Severity_Rate
Is_Highway                                                         
0                   127381                21828               17.14
1                    86663                51047               58.90


In [ ]:
#compare accidents near junctions / not near junctions
print(analyze_factor(df, 'Junction'))

          Accident_Count  High_Severity_Count  High_Severity_Rate
Junction                                                         
0                 203165                66522               32.74
1                  10879                 6353               58.40


In [ ]:
#REGIONAL ANALYSIS

#get all unique region names in dataset
#dropna might be unnecessary, checl
for region in df['Region'].dropna().unique():
  regional_df = df[df['Region'] == region] #filter dataset to only include accidents belonging to current region

  print("REGION:", region)

  print(analyze_factor(
      regional_df, 'Weather_Group' #analyze weather conditions within this region
  ))
  print(analyze_factor(
      regional_df, 'Is_Highway' #analyze highways versus non-highways within this region
  ))
  print(analyze_factor(
      regional_df, 'Junction' #analyze junctions versus no junctions within this region
  ))

REGION: South
               Accident_Count  High_Severity_Count  High_Severity_Rate
Weather_Group                                                         
Clear/Fair              40099                10541               26.29
Cloudy                  50257                15046               29.94
Fog/Low vis              1979                  404               20.41
Rain                     7745                 2714               35.04
Snow/Ice                  254                  113               44.49
Thunderstorm             1714                  689               40.20
Unknown                  2278                  992               43.55
            Accident_Count  High_Severity_Count  High_Severity_Rate
Is_Highway                                                         
0                    67022                10244               15.28
1                    37304                20255               54.30
          Accident_Count  High_Severity_Count  High_Severity_Rate
Junction 

In [ ]:
#COMMUTER ANALYSIS

#compare high-severity percentages across different times of day
print(analyze_factor(df, 'Time_of_Day'))

              Accident_Count  High_Severity_Count  High_Severity_Rate
Time_of_Day                                                          
Evening                13728                 6109               44.50
Evening rush           49427                18619               37.67
Midday                 61868                22872               36.97
Morning rush           72500                19510               26.91
Night                  16521                 5765               34.89


In [ ]:
#compare weekday accidents against weekend accidents
print(analyze_factor(df, 'Is_Weekend'))

            Accident_Count  High_Severity_Count  High_Severity_Rate
Is_Weekend                                                         
0                   192005                60364               31.44
1                    22039                12511               56.77


In [ ]:
#compare nighttime accidents against daytime accidents
print(analyze_factor(df, 'Is_Night'))

          Accident_Count  High_Severity_Count  High_Severity_Rate
Is_Night                                                         
0                 158166                52438               33.15
1                  55878                20437               36.57


#Top Findings:
-**Overall severity:** The dataset contains 214,044 accident records with about 34% classified as high-traffic impact severity.
- **Severity distribution:** Severity 2 was the largest category, accounting for about 65% of all accident records.
- **Regional differenes:** The South had the most accidents (104,326) but the Midwest had the highest overall high-severity rate(42.42%) despite having fewer accidents in the dataset (28,738).
  - 42.42% compared with 29.24% in the South
-**Highway association:** Highways in the datset had higher high-severity rates than non-highways in all four regions.
  - Nationally: 58.90% versus 17.14% for non-highways.
  - Regionally: The Midwest had the highest highway high-severity rate at 68.35% versus 21.92% non-highway.
-Junction association: Accidents with a nearby junction had higher high-severity rates in all four regions.
  - Nationally: 58.40% versus 32.74% without nearby junctions
  - Regionally: The South had the highest junction rate at 68.38%.
-**Weekends patterns:** Weeked accidents had a surprsingly high high-severity rate
  - 56.77% versus 31.44% on weekdays.
  - This finding requires additional investigation to determine whether other factors contribute to the difference.
-**Weather association:** Snow/Ice had the highest national weather-related high-severity rate at 43.88%, followed by Thunderstorms close at 42.85%